# 2-2. Great Expectations 검증 리포트 일반 Notebook

이 notebook은 `vital_signs_valid_degraded.csv`에 데이터 검증 규칙을 적용하고, Great Expectations Demo 산출물을 재생성하는 일반 Jupyter 실습입니다. 수강생은 도구를 설치하고 화면을 꾸미는 사람이 아니라, 검증 실패가 모델 평가 전 판단에 어떤 제한을 남기는지 설명해야 하는 QA 담당자입니다.

핵심 흐름은 `검증 기준 로딩 → degraded 데이터 brief → 품질 검사 실행 → expectation 결과 생성 → artifact 기록 → QA 판단`입니다. JupyterLite에서는 이 결과를 prepared artifact로 읽고, 이 일반 notebook에서는 같은 artifact를 로컬에서 재생성합니다.

| 받은 업무 | 현장 관측값 | 결정 압박 |
| --- | --- | --- |
| 품질 저하 validation 데이터셋의 검증 실패를 재생성하고 QA 해석을 남깁니다 | 2-5에서 feature 품질 실패와 metric 변화가 함께 관측되었습니다 | 지표 하락을 모델 문제로 단정하기 전에 데이터 검증 실패를 artifact로 남겨야 합니다 |

| 이 notebook에서 만드는 증거 | 생성 artifact | 보고서 사용 |
| --- | --- | --- |
| expectation table, validation result table, failure summary, artifact provenance | `chapter_02_expectations.json`, `chapter_02_validation_result.json`, `chapter_02_validation_summary.md`, `chapter_02_data_docs.html` | `조건부 평가`, feature 품질 이슈, 다음 metric 연결 판단 |


### 따라하기 안내

이 notebook은 셀을 위에서 아래로 실행하면서 결과를 해석하는 실습입니다. 코드를 모두 이해하는 것보다, 각 셀이 만드는 근거를 보고 QA 판단으로 바꾸는 것이 중요합니다.

**오늘의 목표:** 검증 rule이 실패했을 때 어떤 column과 조치 후보를 봐야 하는지 확인합니다.

진행 방법은 단순합니다.

1. 안내 문장을 읽고 이 셀이 무엇을 확인하는지 파악합니다.
2. 바로 아래 코드 셀을 실행합니다.
3. 출력에서 핵심 값 1-2개만 고릅니다.
4. 그 값을 보고서에 쓸 수 있는 문장으로 바꿉니다.

마지막에는 다음 형태로 정리합니다.

```text
어떤 expectation이 실패했고, 그 실패가 어떤 QA 확인으로 이어지는지 정리합니다.
```


## 1. 실행 환경과 로컬 재생성 범위

### 1-1. 일반 notebook 실행 기준 고정

이 셀의 판단은 로컬 일반 notebook이 어떤 repo와 package 기준으로 실행되는지 고정하는 것입니다. 이 notebook은 JupyterLite용 축약 경로가 아니라, 로컬 파일 시스템에 artifact를 다시 쓰는 재현 교재입니다.

실행 결과는 `artifacts/great_expectations/` 아래 파일을 갱신합니다. 따라서 보고서에는 “prepared artifact 확인”인지 “이 notebook으로 재생성”인지 구분해서 적어야 합니다.

이 단계에서는 아직 검증을 실행하지 않습니다. 먼저 repo root, package path, 입력 데이터, 출력 디렉터리를 고정합니다.


### 따라하기 안내: 환경 준비

**이 셀에서 할 일**  
GE artifact를 만들기 위한 환경을 준비합니다.

**실행 후 볼 것**  
실행 경로와 package import가 되는지 확인합니다.


In [ ]:
from __future__ import annotations

import json
import sys
from dataclasses import asdict
from pathlib import Path

import pandas as pd


def find_repo_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "packages" / "ai-quality" / "src").exists() and (base / "data").exists():
            return base
    raise RuntimeError("repo root를 찾지 못했습니다.")


ROOT = find_repo_root()
PACKAGE_SRC = ROOT / "packages" / "ai-quality" / "src"
if str(PACKAGE_SRC) not in sys.path:
    sys.path.insert(0, str(PACKAGE_SRC))

DATASET_PATH = ROOT / "data" / "vital_signs_valid_degraded.csv"
OUTPUT_DIR = ROOT / "artifacts" / "great_expectations"

execution_contract = pd.DataFrame(
    [
        {"item": "repo_root", "value": str(ROOT), "qa_use": "artifact 경로를 감사 가능하게 남깁니다."},
        {"item": "dataset", "value": str(DATASET_PATH), "qa_use": "검증 대상 validation 데이터를 고정합니다."},
        {"item": "output_dir", "value": str(OUTPUT_DIR), "qa_use": "재생성되는 검증 artifact 위치를 고정합니다."},
        {"item": "runtime_role", "value": "general_notebook_artifact_regeneration", "qa_use": "Lite artifact inspection과 구분합니다."},
    ]
)

display(execution_contract)


이 출력에서 확인할 핵심은 이 notebook이 파일을 다시 쓰는 일반 실행 경로라는 점입니다. Lite와 달리 로컬 artifact 재생성을 수행하므로, 실행 전후 artifact 내용이 바뀔 수 있습니다.

## 2. 검증 기준과 데이터 brief

### 2-1. schema, quality rule, expectation 기준 확인

이 셀의 판단은 어떤 데이터 기준을 적용할지 먼저 확인하는 것입니다. 검증 결과를 읽기 전에 기준을 보지 않으면, 실패한 규칙이 왜 중요한지 설명할 수 없습니다.

이 notebook은 `dataset_schema.yaml`, `data_quality_rules.yaml`, `great_expectations_rules.yaml`을 로드합니다. schema는 필수 컬럼과 target column을, quality rule은 허용 범위와 label 기준을, expectation은 보고서에 남길 검증 항목과 QA 사유를 정의합니다.

도구 이름보다 중요한 것은 기준의 역할입니다. label 기준 실패는 metric 계산 자체를 흔들고, feature 품질 실패는 score와 prediction 해석에 제한을 남깁니다.


### 따라하기 안내: 설정 파일 읽기

**이 셀에서 할 일**  
schema와 품질 rule 설정을 읽습니다.

**실행 후 볼 것**  
어떤 기준 파일을 쓰는지 봅니다.


In [ ]:
from ai_quality.common.config import load_yaml
from ai_quality.common.paths import config_path
from ai_quality.data_quality.domain.dataset_schema import DatasetSchema
from ai_quality.data_quality.domain.quality_rule import DataQualityRules

schema_config = load_yaml(config_path("validation", "dataset_schema.yaml"))
rules_config = load_yaml(config_path("validation", "data_quality_rules.yaml"))
expectations_config = load_yaml(config_path("validation", "great_expectations_rules.yaml"))

schema = DatasetSchema.from_config(schema_config)
rules = DataQualityRules.from_config(rules_config)
expectations = expectations_config["expectations"]
schema_feature_columns = list(schema.model_feature_columns)

expectation_table = pd.DataFrame(expectations).assign(
    evidence_role=lambda table: table["column"].map(lambda column: "label_contract" if column == schema.target_column else "feature_quality")
)
rule_scope = pd.DataFrame(
    [
        {"rule_source": "dataset_schema.yaml", "observed": f"target={schema.target_column}, required_columns={len(schema.required_columns)}", "qa_use": "평가 입력 구조를 확인합니다."},
        {"rule_source": "data_quality_rules.yaml", "observed": f"valid_ranges={len(rules.valid_ranges)}, allowed_labels={rules.allowed_labels}", "qa_use": "feature 범위와 label 허용값을 확인합니다."},
        {"rule_source": "great_expectations_rules.yaml", "observed": f"expectations={len(expectations)}", "qa_use": "artifact에 남길 검증 기준을 확인합니다."},
    ]
)


### 출력 확인: `rule_scope`

**읽는 순서**  
행 수, 컬럼 수, 데이터셋 이름, 실행 범위를 순서대로 봅니다. 여기서는 아직 품질 판단을 확정하지 않고 “분석 대상이 맞는지”를 확인합니다.

**해석 기준**  
행 수가 예상보다 작으면 JupyterLite sample 또는 fallback sample일 수 있습니다. 컬럼 수나 데이터셋 이름이 기대와 다르면 뒤의 결측, label, metric 해석이 모두 흔들립니다.


In [ ]:
display(rule_scope)


### 출력 확인: `expectation_table`

**읽는 순서**  
expectation type, column, status, unexpected count를 봅니다. 실패한 rule이 어떤 column에 걸렸는지 먼저 확인합니다.

**해석 기준**  
GE 결과는 도구 화면이 아니라 품질 기준의 pass/fail 기록입니다. 실패한 column은 feature 품질 또는 label 기준 문제의 후보가 됩니다.


In [ ]:
display(expectation_table)


이 출력에서 확인할 핵심은 expectation이 단순 체크리스트가 아니라 report evidence라는 점입니다. 각 expectation에는 실패 시 QA가 어떤 제한을 남겨야 하는지 `qa_reason`이 들어 있습니다.

### 2-2. degraded validation 데이터 원본 관찰

이 셀의 판단은 검증 대상 데이터가 어떤 원본 상태인지 확인하는 것입니다. 검증을 실행하기 전에 row count, column count, raw label 분포, feature 결측을 먼저 보면 결과가 어디서 나온 것인지 설명할 수 있습니다.

한 row는 validation classification sample 하나입니다. 이 데이터는 실제 의료 판단이 아니라 생체신호 기반 위험 알림 시스템의 AI QA 예제입니다.

이 단계에서는 아직 expectation 결과를 만들지 않습니다. 원본 데이터 brief를 먼저 고정해 artifact와 dataframe 관측값이 같은 데이터에서 나온 것인지 확인합니다.


### 따라하기 안내: 데이터 로딩

**이 셀에서 할 일**  
검증할 데이터셋을 읽습니다.

**실행 후 볼 것**  
데이터 경로와 행 수를 확인합니다.


In [ ]:
raw_dataframe = pd.read_csv(DATASET_PATH)
preview_columns = [
    column for column in ["patient_id", "timestamp", *schema_feature_columns, schema.target_column]
    if column in raw_dataframe.columns
]

raw_data_brief = pd.DataFrame(
    [
        {"check": "dataset_path", "observed": str(DATASET_PATH), "qa_use": "검증 대상 파일을 고정합니다."},
        {"check": "row_grain", "observed": "one validation classification sample", "qa_use": "검증 실패 비율의 분모를 설명합니다."},
        {"check": "row_count", "observed": raw_dataframe.shape[0], "qa_use": "기대 조건 실패 비율 계산 기준입니다."},
        {"check": "column_count", "observed": raw_dataframe.shape[1], "qa_use": "schema 검증 전 구조를 확인합니다."},
    ]
)
raw_label_distribution = (
    raw_dataframe[schema.target_column]
    .value_counts(dropna=False)
    .rename_axis("raw_label")
    .reset_index(name="count")
    .assign(ratio_pct=lambda table: (table["count"] / len(raw_dataframe) * 100).round(2))
)
raw_missing_snapshot = (
    raw_dataframe.loc[:, schema_feature_columns + [schema.target_column]]
    .isna()
    .sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
    .assign(missing_ratio_pct=lambda table: (table["missing_count"] / len(raw_dataframe) * 100).round(2))
    .sort_values(["missing_count", "column"], ascending=[False, True])
)


### 출력 확인: `raw_data_brief`

**읽는 순서**  
행 수, 컬럼 수, 데이터셋 이름, 실행 범위를 순서대로 봅니다. 여기서는 아직 품질 판단을 확정하지 않고 “분석 대상이 맞는지”를 확인합니다.

**해석 기준**  
행 수가 예상보다 작으면 JupyterLite sample 또는 fallback sample일 수 있습니다. 컬럼 수나 데이터셋 이름이 기대와 다르면 뒤의 결측, label, metric 해석이 모두 흔들립니다.


In [ ]:
display(raw_data_brief)


### 출력 확인: `raw_dataframe.loc[:, preview_columns].head()`

**읽는 순서**  
왼쪽부터 컬럼 이름을 보고, 한 행이 무엇을 의미하는지 확인합니다. 그 다음 feature, label, score, prediction 같은 주요 컬럼이 기대한 위치에 있는지 봅니다.

**해석 기준**  
Preview는 전체 통계가 아니라 샘플 행입니다. 값의 모양, 단위, 컬럼 이름, 변환 후 새 컬럼이 생겼는지를 확인하는 용도입니다.


In [ ]:
display(raw_dataframe.loc[:, preview_columns].head())


### 출력 확인: `raw_label_distribution`

**읽는 순서**  
`high_risk`와 `low_risk`의 count와 ratio를 봅니다. `high_risk`는 positive class이므로 표본 수가 recall 해석의 기본 조건입니다.

**해석 기준**  
두 class가 모두 존재하면 metric 계산은 가능합니다. 한 class가 너무 적으면 accuracy보다 precision/recall과 support를 함께 읽어야 합니다.

**주의할 점**  
class 비율이 균형적이라고 해서 데이터 품질이 모두 좋다는 뜻은 아닙니다. 결측, 범위, label validity는 별도로 봅니다.


In [ ]:
display(raw_label_distribution)


### 출력 확인: `raw_missing_snapshot.head(12)`

**읽는 순서**  
결측 개수와 결측이 발생한 컬럼을 봅니다. feature 결측인지 label 결측인지 구분합니다.

**해석 기준**  
feature 결측은 score 계산을 흔들 수 있고, label 결측은 정답 비교 자체를 어렵게 합니다. 결측률이 낮아도 중요한 컬럼이면 제한 사항입니다.

**보고서 문장 예시**  
“결측은 전체 개수보다 발생 위치를 기준으로 해석했으며, 모델 입력 또는 label 영향 여부를 확인했습니다.”

**주의할 점**  
결측치가 0에 가깝다고 모든 데이터 품질 문제가 해소된 것은 아닙니다. 범위 오류와 label 기준을 계속 확인합니다.


In [ ]:
display(raw_missing_snapshot.head(12))


이 출력에서 확인할 핵심은 검증 결과가 나오기 전에도 `heart_rate` 결측이 관측된다는 점입니다. 따라서 GE artifact의 실패는 도구 내부에서 갑자기 생긴 값이 아니라, 원본 데이터 품질 신호를 기록한 결과입니다.

## 3. 데이터 품질 검사와 validation result 생성

### 3-1. QualityReport 실행 결과 확인

이 셀의 판단은 schema/range/label 검사가 어떤 중간 보고서를 만드는지 확인하는 것입니다. Great Expectations Demo artifact는 이 `QualityReport`에서 expectation별 결과로 변환됩니다.

`QualityReport`는 모델 metric이 아닙니다. 모델 평가 전에 데이터가 어떤 조건으로 흔들리는지 보여주는 입력 품질 evidence입니다.

이 출력에서는 missing column, column quality, range result, label support를 분리해 봅니다. 이 분리가 있어야 label 문제와 feature 품질 문제를 섞지 않습니다.


### 따라하기 안내: 품질 검사 실행

**이 셀에서 할 일**  
저장소 공통 품질 검사 로직을 실행합니다.

**실행 후 볼 것**  
검사 결과의 pass/fail을 봅니다.


In [ ]:
from ai_quality.data_quality.application.inspect_dataset_quality import InspectDatasetQuality
from ai_quality.data_quality.infrastructure.pandas_dataset_reader import PandasDatasetReader

quality_report = InspectDatasetQuality(
    dataset_reader=PandasDatasetReader(schema),
    schema=schema,
    rules=rules,
).run(DATASET_PATH)

column_quality = pd.DataFrame([asdict(item) for item in quality_report.column_quality]).sort_values(
    ["missing_count", "column"], ascending=[False, True]
)
range_results = pd.DataFrame([asdict(item) for item in quality_report.range_results]).sort_values(
    ["invalid_count", "column"], ascending=[False, True]
)
label_support = pd.DataFrame([asdict(quality_report.label_support)]).assign(
    positive_rate_pct=round(quality_report.label_support.positive_rate, 2),
    labeled_count=quality_report.label_support.labeled_count,
)
quality_report_summary = pd.DataFrame(
    [
        {
            "row_count": quality_report.row_count,
            "column_count": quality_report.column_count,
            "missing_columns": list(quality_report.missing_columns),
            "is_evaluation_ready": quality_report.is_evaluation_ready,
            "qa_judgment": "label 기준은 유지되지만 feature 품질 실패를 expectation 결과로 확인해야 합니다."
            if quality_report.is_evaluation_ready else "평가 전 보류 조건이 있는지 세부 실패를 확인해야 합니다.",
        }
    ]
)


### 출력 확인: `quality_report_summary`

**읽는 순서**  
status, pass/fail, note, failed count를 순서대로 봅니다. 어떤 기준이 통과했고 어떤 기준이 제한 사항인지 나눕니다.

**해석 기준**  
gate 결과는 최종 결론이 아니라 다음 단계로 넘어갈 수 있는지 판단하는 checkpoint입니다. fail이나 warning은 보고서 caveat와 next action으로 연결합니다.

**주의할 점**  
pass만 보고 끝내지 않습니다. 어떤 기준을 통과한 것인지, 아직 확인하지 않은 영역은 무엇인지 남겨야 합니다.


In [ ]:
display(quality_report_summary)


### 출력 확인: `column_quality.head(12)`

**읽는 순서**  
표의 첫 번째 열부터 읽고, 어떤 항목이 기준값이고 어떤 항목이 관측값인지 구분합니다.

**해석 기준**  
이 출력은 바로 앞 코드 셀이 만든 단일 근거입니다. 핵심 값 1-2개를 골라 다음 셀의 판단과 연결합니다.

**주의할 점**  
표 전체를 그대로 옮기지 말고, 판단에 필요한 값과 그 의미만 보고서에 남깁니다.


In [ ]:
display(column_quality.head(12))


### 출력 확인: `range_results.loc[range_results["invalid_count"].gt(0)]`

**읽는 순서**  
feature별 최소/최대와 invalid count를 봅니다. 도메인상 가능한 범위를 벗어난 값이 있는지 확인합니다.

**해석 기준**  
범위 오류는 API가 정상 응답해도 score를 비정상적으로 움직일 수 있습니다. 특히 산소포화도, 체온, 혈압 같은 feature는 값의 의미가 중요합니다.

**보고서 문장 예시**  
“range gate에서 허용 범위를 벗어난 feature를 확인했고, score/metric 해석의 제한 사항으로 기록했습니다.”

**주의할 점**  
이상값을 바로 제거한다고 결론 내리지 않습니다. 먼저 수집 오류인지, 단위 문제인지, 실제 rare case인지 확인해야 합니다.


In [ ]:
display(range_results.loc[range_results["invalid_count"].gt(0)])


### 출력 확인: `label_support`

**읽는 순서**  
invalid label, missing label, positive support, negative support를 차례로 봅니다.

**해석 기준**  
invalid 또는 missing label이 있으면 정답 비교가 흔들립니다. positive support가 부족하면 recall 한두 건 차이도 큰 비율 변화로 보일 수 있습니다.

**주의할 점**  
label gate 통과는 feature 품질 통과를 의미하지 않습니다. label은 정답 기준만 확인합니다.


In [ ]:
display(label_support)


이 출력에서 확인할 핵심은 label support가 유지되더라도 feature 품질 실패가 남는다는 점입니다. `is_evaluation_ready`가 넓은 의미에서 True처럼 보여도, feature 결측과 범위 오류는 score와 metric 해석에 제한을 남깁니다.

따라서 다음 셀은 이 중간 보고서를 expectation 결과로 변환합니다. expectation 결과는 수강생이 보고서와 reviewer gate에 붙일 수 있는 artifact 형태입니다.

### 3-2. expectation별 validation result 생성

이 셀의 판단은 품질 검사를 expectation 단위의 성공/실패 표로 바꾸는 것입니다. 이 표가 Great Expectations Demo의 핵심 산출물이며, Lite notebook과 docs가 읽는 원자료입니다.

기대 조건이 실패했다는 사실만으로 모델 평가를 즉시 중단하지 않습니다. 실패 컬럼이 label인지 feature인지, 실패 비율이 어느 정도인지, score와 metric에 어떤 제한을 남기는지 해석해야 합니다.

이 결과에서는 `heart_rate` 결측과 `oxygen_saturation` 범위 오류가 feature 품질 실패로 나타나는지 확인합니다.


### 따라하기 안내: validation result 생성

**이 셀에서 할 일**  
검사 결과를 validation result 형태로 바꿉니다.

**실행 후 볼 것**  
rule별 성공/실패를 확인합니다.

**기록 포인트**  
실패 rule과 column을 적습니다.

**생각해 볼 질문**  
실패한 rule은 어떤 column과 연결되나요?


In [ ]:
from ai_quality.data_quality.application.build_validation_result import build_validation_result

validation_result = build_validation_result(
    report=quality_report,
    expectations=expectations,
    dataset_name=DATASET_PATH.name,
)

validation_result_table = pd.DataFrame(
    [asdict(result) for result in validation_result.expectation_results]
).assign(
    status=lambda table: table["success"].map({True: "pass", False: "fail"}),
    unexpected_ratio_pct=lambda table: table["unexpected_ratio"].round(2),
    evidence_role=lambda table: table["column"].map(lambda column: "label_contract" if column == schema.target_column else "feature_quality"),
)
validation_summary = pd.DataFrame(
    [
        {
            "dataset_name": validation_result.dataset_name,
            "row_count": validation_result.row_count,
            "overall_success": validation_result.success,
            "success_count": validation_result.success_count,
            "failure_count": validation_result.failure_count,
            "qa_judgment": "feature 품질 실패가 있어 조건부 평가와 metric 해석 제한이 필요합니다."
            if validation_result.failure_count else "설정한 expectation 기준은 모두 통과했습니다.",
        }
    ]
)
failure_table = validation_result_table.loc[
    validation_result_table["status"].eq("fail"),
    ["expectation_type", "column", "status", "unexpected_count", "unexpected_ratio_pct", "observed_value", "qa_reason", "evidence_role"],
]


### 출력 확인: `validation_summary`

**읽는 순서**  
status, pass/fail, note, failed count를 순서대로 봅니다. 어떤 기준이 통과했고 어떤 기준이 제한 사항인지 나눕니다.

**해석 기준**  
gate 결과는 최종 결론이 아니라 다음 단계로 넘어갈 수 있는지 판단하는 checkpoint입니다. fail이나 warning은 보고서 caveat와 next action으로 연결합니다.

**보고서 문장 예시**  
“품질 gate 요약에서 통과 항목과 제한 항목을 구분했고, 이후 metric 해석 범위에 반영했습니다.”


In [ ]:
display(validation_summary)


### 출력 확인: `validation_result_table.loc[:, ["expectation_type", "column", "status...`

**읽는 순서**  
expectation type, column, status, unexpected count를 봅니다. 실패한 rule이 어떤 column에 걸렸는지 먼저 확인합니다.

**해석 기준**  
GE 결과는 도구 화면이 아니라 품질 기준의 pass/fail 기록입니다. 실패한 column은 feature 품질 또는 label 기준 문제의 후보가 됩니다.


In [ ]:
display(validation_result_table.loc[:, ["expectation_type", "column", "status", "unexpected_count", "unexpected_ratio_pct", "observed_value", "qa_reason", "evidence_role"]])


### 출력 확인: `failure_table`

**읽는 순서**  
실패한 rule만 모아 column, expectation, 실패 규모를 확인합니다.

**해석 기준**  
이 표는 조치 우선순위를 잡는 데 씁니다. 어떤 column이 반복적으로 실패하면 데이터 수집 또는 전처리 owner에게 확인이 필요합니다.

**보고서 문장 예시**  
“실패 rule 목록에서 조치가 필요한 column을 확인했고, metric 변화의 후보 근거로 남겼습니다.”

**주의할 점**  
실패 목록만 보고 원인을 확정하지 않습니다. 데이터 생성 경로와 metric delta를 함께 봅니다.


In [ ]:
display(failure_table)


이 출력에서 확인할 핵심은 실패가 label 계약이 아니라 feature 품질에 있다는 점입니다. 따라서 지표 계산 형식은 유지되지만, score와 prediction 해석에는 제한을 남깁니다.

이제 이 결과를 파일로 기록합니다. 기록된 JSON/Markdown/HTML은 Lite notebook과 강의 문서에서 prepared artifact로 읽게 됩니다.

## 4. GE Demo artifact 기록과 감사 가능성 확인

### 4-1. expectation/result/summary/Data Docs artifact 생성

이 셀의 판단은 검증 결과가 재현 가능한 artifact로 남는지 확인하는 것입니다. 콘솔 출력만 있으면 reviewer가 나중에 같은 근거를 확인하기 어렵습니다.

생성되는 네 파일은 서로 다른 용도를 가집니다. expectation JSON은 기준, validation result JSON은 원자료, summary Markdown은 보고서 요약, Data Docs HTML은 사람 검토 화면입니다.

이 실행은 파일을 씁니다. 기존 prepared artifact와 같은 값이어야 정상이며, 값이 달라지면 데이터 또는 규칙이 바뀐 것입니다.


### 따라하기 안내: GE artifact 저장

**이 셀에서 할 일**  
expectations, validation result, summary, data docs를 파일로 남깁니다.

**실행 후 볼 것**  
생성된 파일 경로를 확인합니다.

**기록 포인트**  
보고서에 열 수 있는 evidence path를 적습니다.


In [ ]:
from ai_quality.data_quality.infrastructure.great_expectations_artifact_writer import (
    serialize_validation_result,
    write_great_expectations_demo_artifacts,
)

artifacts = write_great_expectations_demo_artifacts(
    expectations=expectations,
    validation_result=validation_result,
    output_dir=OUTPUT_DIR,
)

artifact_table = pd.DataFrame(
    [
        {"artifact": "expectations", "path": str(artifacts.expectations_path), "exists": artifacts.expectations_path.exists(), "qa_use": "적용 기준 확인"},
        {"artifact": "validation_result", "path": str(artifacts.validation_result_path), "exists": artifacts.validation_result_path.exists(), "qa_use": "실패 원자료 확인"},
        {"artifact": "validation_summary", "path": str(artifacts.validation_summary_path), "exists": artifacts.validation_summary_path.exists(), "qa_use": "보고서 요약 확인"},
        {"artifact": "data_docs", "path": str(artifacts.data_docs_path), "exists": artifacts.data_docs_path.exists(), "qa_use": "사람 검토 화면 확인"},
    ]
)
serialized_result = serialize_validation_result(validation_result)
artifact_content_check = pd.DataFrame(
    [
        {
            "check": "serialized_failure_count",
            "observed": serialized_result["failure_count"],
            "expected_use": "Lite artifact와 docs 요약의 실패 조건 수와 맞아야 합니다.",
        },
        {
            "check": "serialized_row_count",
            "observed": serialized_result["row_count"],
            "expected_use": "검증 대상 데이터셋 규모와 맞아야 합니다.",
        },
    ]
)


### 출력 확인: `artifact_table`

**읽는 순서**  
생성된 파일 경로, 존재 여부, 다시 읽기 가능 여부를 봅니다.

**해석 기준**  
화면 출력만 있고 파일이 없으면 reviewer가 같은 근거를 확인하기 어렵습니다. artifact는 재현성과 감사 추적을 위한 증거입니다.


In [ ]:
display(artifact_table)


### 출력 확인: `artifact_content_check`

**읽는 순서**  
생성된 파일 경로, 존재 여부, 다시 읽기 가능 여부를 봅니다.

**해석 기준**  
화면 출력만 있고 파일이 없으면 reviewer가 같은 근거를 확인하기 어렵습니다. artifact는 재현성과 감사 추적을 위한 증거입니다.


In [ ]:
display(artifact_content_check)


이 출력에서 확인할 핵심은 검증 결과가 파일 경로로 남는다는 점입니다. 이 경로가 있어야 2-5 Lite notebook에서 “prepared artifact 확인”이라고 말할 수 있습니다.

생성 artifact가 존재하더라도 QA 판단은 아직 끝나지 않습니다. 실패한 expectation을 모델 평가 전 조건부 평가 또는 보류 기준으로 해석해야 합니다.

### 4-2. 생성 artifact를 다시 읽어 QA 보고 기준으로 확인

이 셀의 판단은 방금 쓴 artifact가 notebook 내부 객체와 같은 내용을 담고 있는지 확인하는 것입니다. 파일을 썼다는 사실만으로 audit-ready라고 볼 수 없으므로, JSON을 다시 읽어 실패 수와 실패 항목을 확인합니다.

이 검사는 수강생이 보고서에 인용할 값을 확인하는 단계입니다. `failure_count`, 실패 컬럼, 실패 비율이 docs와 Lite notebook에서 쓰는 값과 일치해야 합니다.

불일치가 있으면 데이터 파일, 규칙 YAML, artifact 생성 시점을 확인해야 합니다.


### 따라하기 안내: 저장 결과 확인

**이 셀에서 할 일**  
저장된 artifact를 다시 읽어 정상 저장 여부를 확인합니다.

**실행 후 볼 것**  
JSON이 다시 읽히는지 봅니다.


In [ ]:
reloaded_validation_result = json.loads(artifacts.validation_result_path.read_text(encoding="utf-8"))
reloaded_table = pd.DataFrame(reloaded_validation_result["expectation_results"]).assign(
    status=lambda table: table["success"].map({True: "pass", False: "fail"}),
    unexpected_ratio_pct=lambda table: table["unexpected_ratio"].round(2),
)
reloaded_failures = reloaded_table.loc[
    reloaded_table["status"].eq("fail"),
    ["expectation_type", "column", "status", "unexpected_count", "unexpected_ratio_pct", "observed_value", "qa_reason"],
]
reload_check = pd.DataFrame(
    [
        {
            "check": "failure_count_matches",
            "observed": reloaded_validation_result["failure_count"],
            "expected": validation_result.failure_count,
            "status": "pass" if reloaded_validation_result["failure_count"] == validation_result.failure_count else "review",
        },
        {
            "check": "row_count_matches",
            "observed": reloaded_validation_result["row_count"],
            "expected": validation_result.row_count,
            "status": "pass" if reloaded_validation_result["row_count"] == validation_result.row_count else "review",
        },
    ]
)


### 출력 확인: `reload_check`

**읽는 순서**  
생성된 파일 경로, 존재 여부, 다시 읽기 가능 여부를 봅니다.

**해석 기준**  
화면 출력만 있고 파일이 없으면 reviewer가 같은 근거를 확인하기 어렵습니다. artifact는 재현성과 감사 추적을 위한 증거입니다.


In [ ]:
display(reload_check)


### 출력 확인: `reloaded_failures`

**읽는 순서**  
실패한 rule만 모아 column, expectation, 실패 규모를 확인합니다.

**해석 기준**  
이 표는 조치 우선순위를 잡는 데 씁니다. 어떤 column이 반복적으로 실패하면 데이터 수집 또는 전처리 owner에게 확인이 필요합니다.

**보고서 문장 예시**  
“실패 rule 목록에서 조치가 필요한 column을 확인했고, metric 변화의 후보 근거로 남겼습니다.”

**주의할 점**  
실패 목록만 보고 원인을 확정하지 않습니다. 데이터 생성 경로와 metric delta를 함께 봅니다.


In [ ]:
display(reloaded_failures)


이 출력에서 확인할 핵심은 생성한 artifact가 다시 읽어도 같은 실패 항목을 보여준다는 점입니다. 이제 이 결과를 QA 판단과 다음 2-5 metric 연결로 넘길 수 있습니다.

## 5. QA 판단과 다음 notebook handoff

### 5-1. 검증 실패를 조건부 평가 문장으로 바꾸기

마지막 판단은 검증 실패를 도구 출력이 아니라 QA 보고 문장으로 바꾸는 것입니다. `heart_rate` 결측과 `oxygen_saturation` 범위 오류는 feature 품질 실패이므로, 모델 평가를 무조건 중단하기보다 score와 metric 해석에 제한을 붙이는 것이 적절합니다.

다만 label 형식 검증이 통과했다고 해서 label basis 후보가 완전히 사라지는 것은 아닙니다. 허용 label 검증은 값의 형식을 확인하고, 정답 기준 신뢰는 별도 검토 대상입니다.

이 출력은 2-5 데이터-지표 연결 notebook으로 넘길 handoff입니다. 2-5에서는 이 feature 품질 실패가 score/prediction/FP/FN 변화와 같은 방향인지 확인합니다.


### 따라하기 안내: 실패 column 요약

**이 셀에서 할 일**  
실패 column을 조치 후보로 정리합니다.

**실행 후 볼 것**  
실패 column 목록과 요약을 봅니다.


In [ ]:
failed_columns = reloaded_failures["column"].tolist()
feature_failures = reloaded_failures.loc[reloaded_failures["column"].ne(schema.target_column)]
label_failures = reloaded_failures.loc[reloaded_failures["column"].eq(schema.target_column)]

qa_decision = "conditional_evaluation" if len(feature_failures) and label_failures.empty else (
    "hold_evaluation" if not label_failures.empty else "evaluation_ready"
)

chapter_02_ge_packet = pd.DataFrame(
    [
        {
            "evidence": "ge_validation_result",
            "observed": f"failure_count={reloaded_validation_result['failure_count']}, failed_columns={failed_columns}",
            "qa_judgment": "feature 품질 실패가 있어 모델 metric 해석에 제한을 붙입니다.",
            "owner": "QA/Data Engineering",
            "next_action": "2-5에서 score/prediction/FP/FN 변화와 연결합니다.",
        },
        {
            "evidence": "label_contract",
            "observed": f"label_failures={len(label_failures)}",
            "qa_judgment": "허용 label 형식은 유지되지만 label basis review 후보는 별도로 남깁니다.",
            "owner": "Data Owner/QA",
            "next_action": "라벨 반전 후보는 metric 해석 제한 사항으로 기록합니다.",
        },
        {
            "evidence": "artifact_provenance",
            "observed": str(artifacts.validation_result_path),
            "qa_judgment": "검증 결과가 재생성 가능한 artifact로 남았습니다.",
            "owner": "QA",
            "next_action": "Lite notebook에서는 이 artifact를 prepared evidence로 읽습니다.",
        },
    ]
)

handoff_to_data_metric = pd.DataFrame(
    [
        {
            "from_notebook": "great_expectations_lab.ipynb",
            "to_notebook": "data_metric_connection_lab.ipynb",
            "decision": qa_decision,
            "strengthened_candidates": "feature_quality_shift" if len(feature_failures) else "none",
            "open_candidates": "label_basis_review, metric_error_shift",
            "next_question": "feature 품질 실패가 score, prediction, FP/FN 변화와 함께 나타나는가?",
        }
    ]
)

report_sentence = (
    f"Great Expectations Demo 재생성 결과, {DATASET_PATH.name}에서 "
    f"failure_count={reloaded_validation_result['failure_count']}가 확인되었습니다. "
    f"실패 컬럼은 {failed_columns}이며, label 형식 실패는 {len(label_failures)}건입니다. "
    "따라서 모델 평가를 모델 문제로 단정하지 않고, feature 품질 실패가 score와 FP/FN 변화로 이어지는지 2-5에서 확인합니다."
)

print(report_sentence)


### 출력 확인: `chapter_02_ge_packet`

**읽는 순서**  
앞 단계 결과가 어떤 evidence 묶음으로 정리되었는지 봅니다. 다음 장으로 넘길 질문이나 제한 사항도 같이 확인합니다.

**해석 기준**  
packet과 handoff는 최종 보고서 초안에 가깝습니다. 숫자, 조건, 판단, 다음 확인이 함께 들어 있어야 합니다.

**보고서 문장 예시**  
“이 packet을 근거로 다음 장에서 serving 계약 또는 운영 관측 확인으로 넘어갑니다.”

**주의할 점**  
packet이 있다고 분석이 끝난 것은 아닙니다. handoff에 남은 질문을 다음 장에서 계속 확인합니다.


In [ ]:
display(chapter_02_ge_packet)


### 출력 확인: `handoff_to_data_metric`

**읽는 순서**  
앞 단계 결과가 어떤 evidence 묶음으로 정리되었는지 봅니다. 다음 장으로 넘길 질문이나 제한 사항도 같이 확인합니다.

**해석 기준**  
packet과 handoff는 최종 보고서 초안에 가깝습니다. 숫자, 조건, 판단, 다음 확인이 함께 들어 있어야 합니다.

**보고서 문장 예시**  
“이 packet을 근거로 다음 장에서 serving 계약 또는 운영 관측 확인으로 넘어갑니다.”

**주의할 점**  
packet이 있다고 분석이 끝난 것은 아닙니다. handoff에 남은 질문을 다음 장에서 계속 확인합니다.


In [ ]:
display(handoff_to_data_metric)


### 5-2. 실패 시 확인 포인트

실행이 실패하면 먼저 `DATASET_PATH`와 `OUTPUT_DIR`을 확인합니다. 데이터 파일을 찾지 못하면 `labs/prepare_data.py` 또는 데이터 준비 절차가 먼저 필요합니다.

검증 결과가 문서와 다르면 `dataset_schema.yaml`, `data_quality_rules.yaml`, `great_expectations_rules.yaml`이 바뀌었는지 확인합니다. 규칙이 바뀌면 실패 수가 달라지는 것은 정상일 수 있지만, 보고서에는 새 기준으로 재생성했다고 적어야 합니다.

artifact 파일이 생성되지 않으면 `artifacts/great_expectations` 쓰기 권한과 경로를 확인합니다. 이 notebook은 일반 로컬 재생성 경로이므로 JupyterLite에서 실행할 대상이 아닙니다.
